# Topic: Session Security Systems - Session token entropy, HttpOnly/Secure cookie flags, and session timing/revocations
 
## 1. SESSION ID ENTROPY
- **Session ID**: A unique token used to identify an authenticated user session.
- **Security Rule**: Session IDs must possess high entropy (e.g. at least 128 bits of randomness generated using a CSPRNG) to prevent attackers from predicting active session keys via brute-force guessing.
 
## 2. COOKIE SECURITY ATTRIBUTES
- When storing session IDs in browser cookies, three security flags are critical:
  1. **`HttpOnly`**:
     - Prevents client-side scripts (JavaScript `document.cookie`) from accessing the cookie value.
     - **Benefit**: Protects the session cookie from theft if the application is compromised by an XSS vulnerability.
  2. **`Secure`**:
     - Instructs the browser to only transmit the cookie over encrypted HTTPS connections, preventing interception via network sniffing (MitM).
  3. **`SameSite`**:
     - Restricts cookies from being sent on cross-site requests. Options are `Strict`, `Lax`, or `None`.
 
## 3. TIMEOUTS & REVOCATIONS
- **Idle Timeout**: Automatically destroys the session if the user has been inactive for a duration (e.g., 15 minutes).
- **Absolute Timeout**: Forces session termination after a fixed lifespan (e.g., 12 hours) regardless of user activity.
 
---


In [ ]:
import secrets
import time

class SecureSessionManager:
    """Manages secure user sessions with timeout rules and revocation checks."""
    def __init__(self, idle_timeout_sec=3, absolute_timeout_sec=5):
        self.idle_timeout = idle_timeout_sec
        self.absolute_timeout = absolute_timeout_sec
        # Session Store: session_id -> metadata
        self.sessions = {}

    def create_session(self, username):
        """Generates a secure 128-bit entropy session token."""
        # token_urlsafe(16) yields 16 bytes = 128 bits of entropy
        session_id = secrets.token_urlsafe(16)
        now = time.time()
        
        self.sessions[session_id] = {
            "username": username,
            "created_at": now,
            "last_active": now
        }
        return session_id

    def get_session_user(self, session_id):
        """Validates session lifecycle. Returns username if active, else None."""
        session = self.sessions.get(session_id)
        if not session:
            return None
            
        now = time.time()
        
        # 1. Check Absolute Timeout
        if now - session["created_at"] > self.absolute_timeout:
            print(f"[!] SESSION EXPIRED: Absolute timeout reached for user '{session['username']}'")
            self.revoke_session(session_id)
            return None
            
        # 2. Check Idle Timeout
        if now - session["last_active"] > self.idle_timeout:
            print(f"[!] SESSION EXPIRED: Idle timeout reached for user '{session['username']}'")
            self.revoke_session(session_id)
            return None
            
        # Update last active timestamp (reset idle timer)
        session["last_active"] = now
        return session["username"]

    def revoke_session(self, session_id):
        """Secures logouts by explicitly purging session data on the backend."""
        if session_id in self.sessions:
            del self.sessions[session_id]
            print(f"[+] Session revoked successfully.")



In [ ]:
# Testing the Session Lifecycle
print("--- Creating Secure Session ---")
manager = SecureSessionManager(idle_timeout_sec=2, absolute_timeout_sec=4)

session_token = manager.create_session("alice_dev")
print(f"Generated Session ID (high entropy): {session_token}")

# Verify session immediately
print(f"Active User: {manager.get_session_user(session_token)}")  # Expected: alice_dev



In [ ]:
# Test Idle Timeout (Wait 2.5 seconds)
print("\n--- Simulating User Inactivity (Idle Timeout) ---")
time.sleep(2.5)
user = manager.get_session_user(session_token)
print(f"Active User: {user}")  # Expected: None (Revoked due to idle timeout!)



In [ ]:
# Test Absolute Timeout (Wait for absolute expiration limit)
print("\n--- Simulating Active User Exceeding Absolute Timeout ---")
new_token = manager.create_session("bob_admin")

# User interacts every 1 second (keeps idle timer active)
for step in range(1, 6):
    time.sleep(1.0)
    user = manager.get_session_user(new_token)
    print(f"Step {step} | Active User: {user}")
# Expected: Steps 1, 2, 3 return 'bob_admin'.
# Step 4 or 5 triggers absolute timeout and blocks access despite activity!
